In [ ]:
import os

live_root = "/kaggle/input/datasets/username/numerai-live-v52/live.parquet"
weights = "/kaggle/input/datasets/username/numerai-weights-v52-c0096/model_custom_0096.pth"
feat_path = "/kaggle/input/datasets/username/numerai-custom-features/custom_features.json"

print(f"{live_root}, {weights}, {feat_path}")

/kaggle/input/datasets/shuangsong/numerai-live-v52/live.parquet, /kaggle/input/datasets/shuangsong/numerai-weights-v52-c0096/model_custom_0096.pth, /kaggle/input/datasets/shuangsong/numerai-custom-features/custom_features.json


In [13]:
import torch
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, size, dropout_rate=0.5):
        super(ResBlock, self).__init__()
        self.l = nn.Linear(size, size)
        self.bn = nn.BatchNorm1d(size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        identity = x
        out = self.l(x)
        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)
        return out + identity

class NumeraiNet(nn.Module):
    def __init__(self, input_size):
        super(NumeraiNet, self).__init__()
        self.initial_layer = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )
        self.res_blocks = nn.Sequential(
            ResBlock(256, 0.5),
            ResBlock(256, 0.5)
        )
        self.output_layer = nn.Linear(256, 1)

    def forward(self, x):
        x = self.initial_layer(x)
        x = self.res_blocks(x)
        return torch.sigmoid(self.output_layer(x))

In [15]:
import torch
import pandas as pd
import json

with open(feat_path, 'r') as f:
    features = json.load(f)['feature_sets']['custom_features']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NumeraiNet(len(features)).to(device)

model.load_state_dict(torch.load(weights, map_location=device))
model.eval()

live_df = pd.read_parquet(live_root, columns=features)
X_live = torch.from_numpy(live_df[features].values).float().to(device)

with torch.no_grad():
    preds = model(X_live).cpu().numpy().flatten()

submission = pd.DataFrame({
    "id": live_df.index,  
    "prediction": pd.Series(preds).rank(pct=True) 
})
submission["prediction"] = pd.Series(preds).rank(pct=True) 
submission.to_csv("submission.csv", index=False)